Necessory Library


In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import OneHotEncoder, LabelEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, classification_report

In [ ]:
df = pd.read_csv('/content/final_cleaned_filtered_female_crime.csv')

In [ ]:
df.head()

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
df.isnull().sum()

In [ ]:
#convert date column to datetime
df['Date Reported'] = pd.to_datetime(df['Date Reported'], errors='coerce')
df['Date of Occurrence'] = pd.to_datetime(df['Date of Occurrence'], errors='coerce')

Step - 2

Feature Engineering

In [ ]:
#A- Weekend Feature
df['is_weekend'] = df['Date of Occurrence'].dt.dayofweek >= 5

In [ ]:
#B - Month
df['month'] = df['Date of Occurrence'].dt.month

In [ ]:
#C - Week of Year
df['week_of_year'] = df['Date of Occurrence'].dt.isocalendar().week

In [ ]:
#D - Latitude & Longitude Bucketization (FIXED)
df['lat_bucket'] = pd.qcut(df['Latitude'], q=10, labels=False, duplicates='drop')
df['lon_bucket'] = pd.qcut(df['Longitude'], q=10, labels=False, duplicates='drop')

In [ ]:
#E - Crime Keywords --> Binary Features
df['is_rape'] = df['Crime Description'].str.contains('RAPE', case=False, na=False).astype(int)
df['is_assault'] = df['Crime Description'].str.contains('ASSAULT', case=False, na=False).astype(int)
df['is_harass'] = df['Crime Description'].str.contains('HARASS', case=False, na=False).astype(int)

Step - 3
- Drop useless / Leakage Columns

In [ ]:
df = df.drop(['Report Number','Date Reported', 'Date of Occurrence', 'Time of Occurrence', 'Date Case Closed', 'severity_score', 'is_night'], axis=1)

In [ ]:
df.head()

Step - 4
- Encode Target Label

In [ ]:
encoder = LabelEncoder()
df['safety_label'] = encoder.fit_transform(df['safety_label'])

Step - 5
- Features and Target

In [ ]:
X = df.drop('safety_label', axis=1)
y = df['safety_label']

Step - 6
- Numeric & Categorical Features

In [ ]:
from numpy._core import numeric
numeric_features = X.select_dtypes(include=['int64', 'float64']).columns
categorical_features = ['City', 'Crime Domain', 'Weapon Used', 'Victim Gender']

In [ ]:
# ------- Preprocessor for Random Forest (NO SCALING, NO POLY) -------
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median'))
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(drop='first', handle_unknown='ignore'))
])

preprocessor_rf = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ]
)


In [ ]:
# ------- RANDOM FOREST PIPELINE -------
rf_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor_rf),
    ('rf', RandomForestClassifier(
        random_state=42,
        class_weight='balanced'
    ))
])

In [ ]:
X_train, X_test, Y_train, Y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
# ------- GRID SEARCH (Hyperparameter Tuning) -------
param_grid_rf = {
    'rf__n_estimators': [100, 200],
    'rf__max_depth': [10, 20, None],
    'rf__min_samples_leaf': [1, 2, 4]
}
grid_rf = GridSearchCV(
    rf_pipeline,
    param_grid_rf,
    cv=4,
    scoring='accuracy',
    n_jobs=-1
)

grid_rf.fit(X_train, Y_train)

print("Best Parameters:", grid_rf.best_params_)

best_rf_model = grid_rf.best_estimator_

In [ ]:
# ------- Evaluation -------
y_pred_train_rf = best_rf_model.predict(X_train)
y_pred_test_rf = best_rf_model.predict(X_test)

print("\nRANDOM FOREST RESULTS")
print("Training Accuracy:", accuracy_score(Y_train, y_pred_train_rf))
print("Testing Accuracy:", accuracy_score(Y_test, y_pred_test_rf))

print("\nClassification Report:")
print(classification_report(Y_test, y_pred_test_rf))

In [ ]:
print("\nData Used Summary: ")
print("Original Test Data: ", len(df))
print("Train Data: ", len(X_train))
print("Test Data: ", len(X_test))

In [ ]:
input_data = {
    # Original categorical features
    "City": "Delhi",
    "Crime Domain": "Public Place",
    "Weapon Used": "Knife",
    "Victim Gender": "Female",

    # Original numeric features
    "Victim Age": 23,
    "Crime Code": 102,
    "Police Deployed": 2,
    "Latitude": 28.7041,
    "Longitude": 77.1025,
    "hour_of_day": 22,   # 10 PM

    # Feature-engineered columns
    "is_weekend": 0,
    "month": 5,
    "week_of_year": 18,
    "lat_bucket": 5,
    "lon_bucket": 4,

    # Crime description keywords
    "is_rape": 0,
    "is_assault": 1,
    "is_harass": 1
}

# Convert to DataFrame
input_df = pd.DataFrame([input_data])

#predict
prediction = best_rf_model.predict(input_df)
prediction_prob = best_rf_model.predict_proba(input_df)

# Output
print("Prediction Output: ", prediction)

if(prediction[0] == 0):
  print("Prediction: ?Unsafe Area")
else:
  print("Prediction: ?Safe Area")

print("Probability (Unsafe/Safe): ", prediction_prob)

In [ ]:
# =====================================================
# ROC?AUC Curve for RANDOM FOREST
# =====================================================

import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, auc

# Predict probability (positive class = 1)
y_prob_rf = best_rf_model.predict_proba(X_test)[:, 1]

# Compute ROC curve
fpr_rf, tpr_rf, _ = roc_curve(Y_test, y_prob_rf)

# Compute AUC
auc_rf = auc(fpr_rf, tpr_rf)

# Plot ROC Curve
plt.figure(figsize=(8,6))
plt.plot(fpr_rf, tpr_rf, label=f"Random Forest (AUC = {auc_rf:.3f})", color='green')
plt.plot([0, 1], [0, 1], 'k--', label="Random Guess")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve - Random Forest")
plt.legend()
plt.grid(True)
plt.show()
